In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Q(3-a) measure bias2 , variance n test accuracy

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_olivetti_faces
from sklearn.tree import DecisionTreeClassifier
from pathlib import Path
import shutil



In [3]:
# 1.load data n resuse saved split

faces = fetch_olivetti_faces()
X = faces.images.reshape(len(faces.images), -1)
y = faces.target

BASE_DIR = Path.cwd()
matches = list(BASE_DIR.rglob("olivetti_split.npz"))

if not matches:
    raise FileNotFoundError(
        "olivetti_split.npz not found in the current project folder. "
        "Place the file in your project directory or a subfolder."
    )

split_path = matches[0]
print("Split file found:", split_path.relative_to(BASE_DIR))

split = np.load(split_path)

train_indices = split["train_indices"]
test_indices = split["test_indices"]
seed = int(split["seed"])

X_train = X[train_indices]
y_train = y[train_indices]
X_test = X[test_indices]
y_test = y[test_indices]

classes = np.arange(40)
N = len(X_test)

print("Original training samples:", len(X_train))
print("Clean test samples:", N)
print("Number of classes:", len(classes))

downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /root/scikit_learn_data
Split file found: drive/MyDrive/Machine Learning IITJ/Assignment-2/olivetti_split.npz
Original training samples: 240
Clean test samples: 80
Number of classes: 40


In [4]:
# frozen setting selected in q1 n q2

TREE_DEPTH = None
MIN_SAMPLES_LEAF = 4
B_FINAL = 50

BOOST_DEPTH = 3
BOOST_ROUNDS = 100

R = 30

q3_rng = np.random.default_rng(seed)

# create one resampled training set(sample 6 imgs with replacement within each identity)
def resample_training_set(rng):

  selected_indices = []

  for identity in classes:

    identity_positions = np.where(y_train == identity)[0]

    sampled_positions = rng.choice(
        identity_positions,
        size=6,
        replace=True
    )

    selected_indices.extend(sampled_positions)

  selected_indices = np.array(selected_indices)
  rng.shuffle(selected_indices)

  return (
      X_train[selected_indices],
      y_train[selected_indices]
  )


In [5]:
# samme boosting implementation(same q2)

def train_samme(X_fit, y_fit, depth, n_rounds, random_seed):

  model_classes = np.unique(y_fit)
  K = len(model_classes)
  n = len(y_fit)

  weights = np.ones(n) / n
  learners = []
  alphas = []

  for t in range(n_rounds):

    tree = DecisionTreeClassifier(
        max_depth = depth,
        random_state = random_seed + t
    )

    tree.fit(X_fit, y_fit, sample_weight = weights)

    pred = tree.predict(X_fit)
    wrong = pred != y_fit

    error = np.sum(weights[wrong])

    if error >= (1 - (1 / K)):
      break

    if error <= 0:
      alpha = np.log((1 - 1e-12) / 1e-12) + np.log(K - 1)
      learners.append(tree)
      alphas.append(alpha)
      break

    alpha = np.log((1 - error) / error) + np.log(K - 1)

    learners.append(tree)
    alphas.append(alpha)

    weights[wrong] *= np.exp(alpha)
    weights /= np.sum(weights)

  return {
    "learners": learners,
    "alphas": np.asarray(alphas),
    "classes": model_classes
  }

In [6]:
# convert each model's pred into classs vector{all o/p vectors have 40 entries per imgs}

def single_tree_vectors(model, X_eval):

  probabilities = model.predict_proba(X_eval)
  vectors = np.zeros((len(X_eval), 40))

  for j, class_label in enumerate(model.classes_):
    vectors[:, int(class_label)] = probabilities[:, j]

  return vectors


In [7]:
def bagging_vectors(trees, X_eval):

  predictions = np.array([tree.predict(X_eval) for tree in trees])

  vectors = np.zeros((len(X_eval), 40))

  for tree_predictions in predictions:
    for i, label in enumerate(tree_predictions):
      vectors[i, int(label)] += 1

  vectors /= len(trees)

  return vectors

In [8]:
def boosting_vectors(model, X_eval):

  vectors = np.zeros((len(X_eval), 40))
  total_alpha = np.sum(model["alphas"])

  for tree, alpha in zip(
      model["learners"],
      model["alphas"]
  ):

    predictions = tree.predict(X_eval)

    for i, label in enumerate(predictions):
      vectors[i, int(label)] += alpha

  if total_alpha > 0:
    vectors /= total_alpha

  return vectors


In [9]:
# store 30 experiments

models = {
    "Single Tree" : [],
    "Bagging (50 trees)" : [],
    "SAMME Boosting" : []
}

accuracy_history = {
    name : [] for name in models
}

prediction_history = {
    name : [] for name in models
}

for r in range(R):

  print(f"Running repetition {r + 1}/{R}")

  X_resampled, y_resampled = resample_training_set(q3_rng)

  assert len(X_resampled) == 240
  assert len(y_resampled) == 240

  repetition_seed = seed + 1000 * (r + 1)

  # A.single DT

  single_tree = DecisionTreeClassifier(
    max_depth=TREE_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=repetition_seed
  )

  single_tree.fit(X_resampled, y_resampled)

  # B-bagging - 50 trees
  # each tree bootstraping the curr resampled traing set

  bag_rng = np.random.default_rng(repetition_seed)
  bagged_trees = []
  for b in range(B_FINAL):

    bootstrap_indices = bag_rng.choice(
        len(X_resampled),
        size=len(X_resampled),
        replace=True
    )

    tree = DecisionTreeClassifier(
        max_depth=TREE_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        random_state=repetition_seed + b + 1
    )

    tree.fit(
        X_resampled[bootstrap_indices],
        y_resampled[bootstrap_indices]
    )

    bagged_trees.append(tree)

  # C-samme boosting: d = 3, 100 rounds

  boost_model = train_samme(
      X_resampled,
      y_resampled,
      depth = BOOST_DEPTH,
      n_rounds = BOOST_ROUNDS,
      random_seed = repetition_seed
  )

  # predict the same clean test set

  tree_vec = single_tree_vectors(single_tree, X_test)
  bag_vec = bagging_vectors(bagged_trees, X_test)
  boost_vec = boosting_vectors(boost_model, X_test)

  repetition_predictions = {
      "Single Tree" : tree_vec,
      "Bagging (50 trees)" : bag_vec,
      "SAMME Boosting" : boost_vec
  }

  for name, vectors in repetition_predictions.items():

    # Save complete class vectors, not just class labels
    prediction_history[name].append(vectors)

    predicted_labels = classes[np.argmax(vectors, axis=1)]

    accuracy = np.mean(predicted_labels == y_test)
    accuracy_history[name].append(accuracy)

    models[name].append(None)  # Predictions are stored above.


Running repetition 1/30
Running repetition 2/30
Running repetition 3/30
Running repetition 4/30
Running repetition 5/30
Running repetition 6/30
Running repetition 7/30
Running repetition 8/30
Running repetition 9/30
Running repetition 10/30
Running repetition 11/30
Running repetition 12/30
Running repetition 13/30
Running repetition 14/30
Running repetition 15/30
Running repetition 16/30
Running repetition 17/30
Running repetition 18/30
Running repetition 19/30
Running repetition 20/30
Running repetition 21/30
Running repetition 22/30
Running repetition 23/30
Running repetition 24/30
Running repetition 25/30
Running repetition 26/30
Running repetition 27/30
Running repetition 28/30
Running repetition 29/30
Running repetition 30/30


In [10]:
print(repetition_predictions.keys())
print(prediction_history.keys())
print(accuracy_history.keys())
print(models.keys())

dict_keys(['Single Tree', 'Bagging (50 trees)', 'SAMME Boosting'])
dict_keys(['Single Tree', 'Bagging (50 trees)', 'SAMME Boosting'])
dict_keys(['Single Tree', 'Bagging (50 trees)', 'SAMME Boosting'])
dict_keys(['Single Tree', 'Bagging (50 trees)', 'SAMME Boosting'])


In [11]:
# CALCULATE BIAS SQUARED, VARIANCE AND ACCURACY STATISTICS

results = []

# Correct one-hot target vector for each clean test image
one_hot_targets = np.eye(40)[y_test]

for name in models:

    # Shape: (30 repetitions, N test images, 40 classes)
    predictions = np.asarray(prediction_history[name])

    # Mean class vector over all repetitions for each test image
    mean_predictions = np.mean(predictions, axis=0)

    # Bias squared:
    # (1/N) sum_i || mean_prediction_i - one_hot_target_i ||^2
    bias_squared = np.mean(
        np.sum(
            (mean_predictions - one_hot_targets) ** 2,
            axis=1
        )
    )

    # Variance:
    # (1/(N*R)) sum_i,r || prediction_ri - mean_prediction_i ||^2
    squared_deviations = np.sum(
        (predictions - mean_predictions[None, :, :]) ** 2,
        axis=2
    )

    variance = np.mean(squared_deviations)

    accuracies = np.asarray(accuracy_history[name])

    results.append({
        "Model": name,
        "Bias Squared": bias_squared,
        "Variance": variance,
        "Mean Test Accuracy": np.mean(accuracies),
        "Std Test Accuracy": np.std(accuracies, ddof=0)
    })



In [12]:
# DISPLAY AND SAVE THE FINAL TABLE

results_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("Q3(a): MEASURED BIAS-VARIANCE RESULTS")
print("=" * 75)

print(
    results_df.to_string(
        index=False,
        formatters={
            "Bias Squared": "{:.6f}".format,
            "Variance": "{:.6f}".format,
            "Mean Test Accuracy": "{:.4f}".format,
            "Std Test Accuracy": "{:.4f}".format
        }
    )
)

results_df.to_csv("Q3a_bias_variance_results.csv", index=False)

# Keep the prediction vectors available for later analysis if needed.
np.savez_compressed(
    "Q3a_prediction_history.npz",
    **{
        name.replace(" ", "_").replace("(", "").replace(")", ""):
        np.asarray(prediction_history[name])
        for name in prediction_history
    }
)

print("\nSaved results to Q3a_bias_variance_results.csv")
print("Saved class prediction vectors to Q3a_prediction_history.npz")



Q3(a): MEASURED BIAS-VARIANCE RESULTS
             Model Bias Squared Variance Mean Test Accuracy Std Test Accuracy
       Single Tree     0.374528 0.635472             0.4950            0.0456
Bagging (50 trees)     0.524167 0.032239             0.7550            0.0440
    SAMME Boosting     0.914906 0.015014             0.1963            0.0499

Saved results to Q3a_bias_variance_results.csv
Saved class prediction vectors to Q3a_prediction_history.npz


In [13]:
# Q3(3-b)

print('ndwude')



ndwude


In [14]:
#  Q3(b): Effect of tree depth on bias, variance and bagging

R_B = 30
B_B = 100

# Three deepth settings
depth_settings = {
    "Shallow (depth=2)": 2,
    "Selected (unrestricted)": None,
    "Deep (depth=20)": 20
}

# Use the same leaf setting selected in Q1
MIN_LEAF_B = MIN_SAMPLES_LEAF

# Independent, reproducible resampling sequence for Q3(b)
q3b_rng = np.random.default_rng(seed + 30000)

# Store predictions and accuracies for each configuration
q3b_names = []

for depth_name in depth_settings:
    q3b_names.append(f"Single Tree - {depth_name}")
    q3b_names.append(f"Bagging 100 - {depth_name}")

q3b_predictions = {name: [] for name in q3b_names}
q3b_accuracies = {name: [] for name in q3b_names}

for r in range(R_B):
    print(f"Q3(b): Running repetition {r + 1}/{R_B}")

    # One resampled dataset, shared across all six configurations
    X_resampled, y_resampled = resample_training_set(q3b_rng)

    repetition_seed = seed + 40000 + r * 1000

    for depth_name, depth in depth_settings.items():

        # 1. Single decision tree
        single_tree = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=MIN_LEAF_B,
            random_state=repetition_seed
        )
        single_tree.fit(X_resampled, y_resampled)

        single_name = f"Single Tree - {depth_name}"
        single_vec = single_tree_vectors(single_tree, X_test)

        q3b_predictions[single_name].append(single_vec)
        single_labels = classes[np.argmax(single_vec, axis=1)]
        q3b_accuracies[single_name].append(
            np.mean(single_labels == y_test)
        )

        # 2. Bagging ensemble with 100 trees
        bag_rng = np.random.default_rng(
            repetition_seed + 100
        )
        bagged_trees = []

        for b in range(B_B):
            bootstrap_indices = bag_rng.choice(
                len(X_resampled),
                size=len(X_resampled),
                replace=True
            )

            tree = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_leaf=MIN_LEAF_B,
                random_state=repetition_seed + b + 101
            )

            tree.fit(
                X_resampled[bootstrap_indices],
                y_resampled[bootstrap_indices]
            )

            bagged_trees.append(tree)

        bag_name = f"Bagging 100 - {depth_name}"
        bag_vec = bagging_vectors(bagged_trees, X_test)

        q3b_predictions[bag_name].append(bag_vec)
        bag_labels = classes[np.argmax(bag_vec, axis=1)]
        q3b_accuracies[bag_name].append(
            np.mean(bag_labels == y_test)
        )

# Calculate bias squared, variance and accuracy statistics
one_hot_targets = np.eye(40)[y_test]
q3b_results = []

for name in q3b_names:
    # Shape: (30 repetitions, test samples, 40 classes)
    pred = np.asarray(q3b_predictions[name])

    mean_pred = np.mean(pred, axis=0)

    bias_squared = np.mean(
        np.sum((mean_pred - one_hot_targets) ** 2, axis=1)
    )

    variance = np.mean(
        np.sum((pred - mean_pred[None, :, :]) ** 2, axis=2)
    )

    accuracies = np.asarray(q3b_accuracies[name])

    q3b_results.append({
        "Model": name,
        "Bias squared": bias_squared,
        "Variance": variance,
        "Mean test accuracy": np.mean(accuracies),
        "SD test accuracy": np.std(accuracies, ddof=0)
    })

q3b_table = pd.DataFrame(q3b_results)

print("\nQ3(b): Results")
display(q3b_table.round(4))


Q3(b): Running repetition 1/30
Q3(b): Running repetition 2/30
Q3(b): Running repetition 3/30
Q3(b): Running repetition 4/30
Q3(b): Running repetition 5/30
Q3(b): Running repetition 6/30
Q3(b): Running repetition 7/30
Q3(b): Running repetition 8/30
Q3(b): Running repetition 9/30
Q3(b): Running repetition 10/30
Q3(b): Running repetition 11/30
Q3(b): Running repetition 12/30
Q3(b): Running repetition 13/30
Q3(b): Running repetition 14/30
Q3(b): Running repetition 15/30
Q3(b): Running repetition 16/30
Q3(b): Running repetition 17/30
Q3(b): Running repetition 18/30
Q3(b): Running repetition 19/30
Q3(b): Running repetition 20/30
Q3(b): Running repetition 21/30
Q3(b): Running repetition 22/30
Q3(b): Running repetition 23/30
Q3(b): Running repetition 24/30
Q3(b): Running repetition 25/30
Q3(b): Running repetition 26/30
Q3(b): Running repetition 27/30
Q3(b): Running repetition 28/30
Q3(b): Running repetition 29/30
Q3(b): Running repetition 30/30

Q3(b): Results


,Model,Bias squared,Variance,Mean test accuracy,SD test accuracy
0,Single Tree - Shallow (depth=2),0.9189,0.0363,0.0550,0.0160
1,Bagging 100 - Shallow (depth=2),0.9319,0.0113,0.1463,0.0634
2,Single Tree - Selected (unrestricted),0.4063,0.6720,0.4608,0.0518
3,Bagging 100 - Selected (unrestricted),0.5304,0.0258,0.7617,0.0432
4,Single Tree - Deep (depth=20),0.5864,0.3210,0.3146,0.0462
5,Bagging 100 - Deep (depth=20),0.6261,0.0228,0.6671,0.0423


In [15]:

variance_comparison = []

for depth_name in depth_settings:
    single_name = f"Single Tree - {depth_name}"
    bag_name = f"Bagging 100 - {depth_name}"

    single_row = q3b_table[
        q3b_table["Model"] == single_name
    ].iloc[0]

    bag_row = q3b_table[
        q3b_table["Model"] == bag_name
    ].iloc[0]

    single_var = single_row["Variance"]
    bag_var = bag_row["Variance"]

    removed = single_var - bag_var

    percent_removed = (
        100 * removed / single_var
        if single_var > 0 else np.nan
    )

    variance_comparison.append({
        "Depth setting": depth_name,
        "Single-tree variance": single_var,
        "Bagging variance": bag_var,
        "Variance removed": removed,
        "Variance reduction (%)": percent_removed,
        "Single-tree accuracy": single_row["Mean test accuracy"],
        "Bagging accuracy": bag_row["Mean test accuracy"],
        "Accuracy improvement": (
            bag_row["Mean test accuracy"]
            - single_row["Mean test accuracy"]
        )
    })

variance_table = pd.DataFrame(variance_comparison)

print("\nVariance reduction and accuracy improvement")
display(variance_table.round(4))



Variance reduction and accuracy improvement


,Depth setting,Single-tree variance,Bagging variance,Variance removed,Variance reduction (%),Single-tree accuracy,Bagging accuracy,Accuracy improvement
0,Shallow (depth=2),0.0363,0.0113,0.0249,68.7629,0.0550,0.1463,0.0913
1,Selected (unrestricted),0.6720,0.0258,0.6463,96.1661,0.4608,0.7617,0.3008
2,Deep (depth=20),0.3210,0.0228,0.2983,92.9059,0.3146,0.6671,0.3525


In [16]:

# Q3(c): Is the deepest tree always best for bagging?

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

B_C = 100
RANDOM_SEED_C = seed + 50000

# Use the same depth settings as Q3(b)
q3c_rng = np.random.default_rng(RANDOM_SEED_C)

# One reproducible resampled training set
X_resampled, y_resampled = resample_training_set(q3c_rng)

q3c_rows = []

for depth_name, depth in depth_settings.items():

    # --------------------------------------------------
    # 1. Fit a single tree
    # --------------------------------------------------
    single_tree = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=MIN_LEAF_B,
        random_state=RANDOM_SEED_C
    )

    single_tree.fit(X_resampled, y_resampled)

    # Leaf occupancy for the single tree
    single_leaf_ids = np.where(
        single_tree.tree_.children_left == -1
    )[0]

    single_leaf_counts = (
        single_tree.tree_.n_node_samples[single_leaf_ids]
    )

    single_mean_occupancy = np.mean(single_leaf_counts)

    single_singleton_fraction = np.mean(
        single_leaf_counts == 1
    )

    # --------------------------------------------------
    # 2. Fit 100 trees using bootstrap samples
    # --------------------------------------------------
    bag_rng = np.random.default_rng(
        RANDOM_SEED_C + 100
    )

    bagged_trees = []
    bag_leaf_occupancies = []
    bag_singleton_fractions = []

    for b in range(B_C):

        indices = bag_rng.choice(
            len(X_resampled),
            size=len(X_resampled),
            replace=True
        )

        tree = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=MIN_LEAF_B,
            random_state=RANDOM_SEED_C + b + 101
        )

        tree.fit(
            X_resampled[indices],
            y_resampled[indices]
        )

        bagged_trees.append(tree)

        leaf_ids = np.where(
            tree.tree_.children_left == -1
        )[0]

        leaf_counts = tree.tree_.n_node_samples[leaf_ids]

        bag_leaf_occupancies.append(np.mean(leaf_counts))
        bag_singleton_fractions.append(
            np.mean(leaf_counts == 1)
        )

    # --------------------------------------------------
    # 3. Pairwise disagreement on the fixed clean test set
    # --------------------------------------------------
    # Shape: (100 trees, 80 test images)
    predictions = np.array([
        tree.predict(X_test) for tree in bagged_trees
    ])

    # For each test image, calculate the fraction of
    # tree pairs that predict different identities.
    disagreements = []

    for i in range(len(X_test)):

        _, counts = np.unique(
            predictions[:, i],
            return_counts=True
        )

        agreeing_pairs = np.sum(counts * (counts - 1)) / 2

        total_pairs = B_C * (B_C - 1) / 2

        disagreements.append(
            1 - agreeing_pairs / total_pairs
        )

    mean_disagreement = np.mean(disagreements)

    # --------------------------------------------------
    # 4. Retrieve measurements from Q3(b)
    # --------------------------------------------------
    single_name = f"Single Tree - {depth_name}"
    bag_name = f"Bagging 100 - {depth_name}"

    single_row = q3b_table[
        q3b_table["Model"] == single_name
    ].iloc[0]

    bag_row = q3b_table[
        q3b_table["Model"] == bag_name
    ].iloc[0]

    single_acc = single_row["Mean test accuracy"]
    bag_acc = bag_row["Mean test accuracy"]

    single_var = single_row["Variance"]
    bag_var = bag_row["Variance"]

    # --------------------------------------------------
    # 5. Store the measured comparisons
    # --------------------------------------------------
    q3c_rows.append({
        "Depth": depth_name,

        "Single accuracy": single_acc,
        "Bagged accuracy": bag_acc,
        "Accuracy improvement": bag_acc - single_acc,

        "Single bias^2": single_row["Bias squared"],
        "Bagged bias^2": bag_row["Bias squared"],

        "Single variance": single_var,
        "Bagged variance": bag_var,
        "Variance reduction": single_var - bag_var,

        "Single mean leaf occupancy": single_mean_occupancy,
        "Single singleton leaf fraction": single_singleton_fraction,

        "Bagged mean leaf occupancy": np.mean(
            bag_leaf_occupancies
        ),
        "Bagged singleton leaf fraction": np.mean(
            bag_singleton_fractions
        ),

        "Pairwise disagreement": mean_disagreement
    })

q3c_table = pd.DataFrame(q3c_rows)

print("Q3(c): MEASURED DEPTH COMPARISON")
display(q3c_table.round(4))

# --------------------------------------------------
# 6. Identify the largest measured accuracy improvement
# --------------------------------------------------
best_improvement = q3c_table.loc[
    q3c_table["Accuracy improvement"].idxmax()
]

deepest = q3c_table.iloc[-1]

print("\nLargest accuracy improvement from bagging:")
print("Depth:", best_improvement["Depth"])
print(
    "Accuracy improvement:",
    round(best_improvement["Accuracy improvement"], 4)
)

print("\nDeepest tree:")
print("Depth:", deepest["Depth"])
print(
    "Accuracy improvement:",
    round(deepest["Accuracy improvement"], 4)
)

if best_improvement["Depth"] == deepest["Depth"]:
    print(
        "\nIn these measurements, the deepest tree had "
        "the largest accuracy improvement."
    )
else:
    print(
        "\nIn these measurements, the deepest tree did NOT "
        "have the largest accuracy improvement."
    )


Q3(c): MEASURED DEPTH COMPARISON


,Depth,Single accuracy,Bagged accuracy,Accuracy improvement,Single bias^2,Bagged bias^2,Single variance,Bagged variance,Variance reduction,Single mean leaf occupancy,Single singleton leaf fraction,Bagged mean leaf occupancy,Bagged singleton leaf fraction,Pairwise disagreement
0,Shallow (depth=2),0.0550,0.1463,0.0913,0.9189,0.9319,0.0363,0.0113,0.0249,80.0000,0.0,77.0000,0.0,0.9621
1,Selected (unrestricted),0.4608,0.7617,0.3008,0.4063,0.5304,0.6720,0.0258,0.6463,6.0000,0.0,6.1125,0.0,0.8373
2,Deep (depth=20),0.3146,0.6671,0.3525,0.5864,0.6261,0.3210,0.0228,0.2983,10.9091,0.0,9.1404,0.0,0.8764



Largest accuracy improvement from bagging:
Depth: Deep (depth=20)
Accuracy improvement: 0.3525

Deepest tree:
Depth: Deep (depth=20)
Accuracy improvement: 0.3525

In these measurements, the deepest tree had the largest accuracy improvement.


In [17]:

# Q3(d): Final comparison table

import pandas as pd

final_table = pd.DataFrame([
    {
        "Model": "Single Tree",
        "Clean Test Accuracy": 0.5625,
        "Masked Test Accuracy": 0.2750,
        "Bias Squared": 0.374528,
        "Variance": 0.635472
    },
    {
        "Model": "Bagging (50 trees)",
        "Clean Test Accuracy": 0.9000,
        "Masked Test Accuracy": 0.7375,
        "Bias Squared": 0.524167,
        "Variance": 0.032239
    },
    {
        "Model": "SAMME Boosting",
        "Clean Test Accuracy": 0.3875,
        "Masked Test Accuracy": 0.3875,
        "Bias Squared": 0.914906,
        "Variance": 0.015014
    }
])

print("Q3(d): FINAL MODEL COMPARISON")
display(final_table.round(6))

# Save the final comparison table
final_table.to_csv("Q3d_final_comparison.csv", index=False)
print("Table saved as Q3d_final_comparison.csv")


Q3(d): FINAL MODEL COMPARISON


,Model,Clean Test Accuracy,Masked Test Accuracy,Bias Squared,Variance
0,Single Tree,0.5625,0.2750,0.374528,0.635472
1,Bagging (50 trees),0.9000,0.7375,0.524167,0.032239
2,SAMME Boosting,0.3875,0.3875,0.914906,0.015014


Table saved as Q3d_final_comparison.csv
